
# Detección Avanzada del "Momento Óptimo"
Este notebook explora soluciones estadísticas y probabilísticas para detectar el momento ideal de intervención comercial, yendo más allá de reglas fijas de negocio.

## Objetivo
Detectar el "Momento Óptimo" (MO) para tres perfiles clave:
1. **Clientes Leales**: Anticipar la reposición antes de que se agote el stock.
2. **Clientes Promiscuos**: Capturar la ventana de compra frente a la competencia.
3. **Clientes en Riesgo**: Detectar el deterioro del patrón de compra (fuga silenciosa).

---


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Carga de datos
df = pd.read_csv('data/master_commodities.csv')
df['Fecha'] = pd.to_datetime(df['Fecha'])
df = df[df['es_devolucion'] == 0] # Solo ventas netas

print(f"Dataset cargado: {len(df):,} transacciones")



## Solución A: Modelado Probabilístico del Tiempo entre Compras
En lugar de usar un "promedio + X días", modelamos la distribución real de los intervalos entre compras (Inter-arrival Times) usando una **Distribución Gamma**. 

Esto nos permite calcular la **Probabilidad Acumulada de Compra**. El "Momento Óptimo" es cuando la probabilidad de que el cliente *ya debería haber comprado* supera un umbral (ej. 80%).


In [ ]:

def analyze_client_intervals(client_id, family='Anestesia'):
    client_df = df[(df['Id_Cliente'] == client_id) & (df['Familia_Potencial'] == family)]
    dates = client_df['Fecha'].sort_values().unique()
    
    if len(dates) < 3:
        return None
    
    intervals = np.diff(dates.astype('datetime64[D]')).astype(float)
    
    # Ajuste de distribución Gamma
    # alpha: shape, loc, beta: scale
    try:
        alpha, loc, beta = stats.gamma.fit(intervals)
    except:
        return None
    
    return {
        'intervals': intervals,
        'params': (alpha, loc, beta),
        'last_date': dates[-1]
    }

def plot_optimal_window(client_id, family='Anestesia'):
    res = analyze_client_intervals(client_id, family)
    if not res: return print("Datos insuficientes")
    
    intervals = res['intervals']
    alpha, loc, beta = res['params']
    last_date = res['last_date']
    
    x = np.linspace(0, max(intervals) * 2.5, 100)
    pdf = stats.gamma.pdf(x, alpha, loc, beta)
    cdf = stats.gamma.cdf(x, alpha, loc, beta)
    
    fig, ax1 = plt.subplots()
    
    ax1.plot(x, pdf, 'b-', label='Probabilidad (PDF)')
    ax1.fill_between(x, pdf, color='blue', alpha=0.1)
    ax1.set_xlabel('Días desde última compra')
    ax1.set_ylabel('Densidad', color='b')
    
    ax2 = ax1.twinx()
    ax2.plot(x, cdf, 'r--', label='Acumulada (CDF)')
    ax2.axhline(0.8, color='orange', linestyle=':', label='Umbral 80%')
    ax2.set_ylabel('Probabilidad Acumulada', color='r')
    
    # Calcular días para el momento óptimo
    day_optimo = stats.gamma.ppf(0.8, alpha, loc, beta)
    ax2.axvline(day_optimo, color='green', lw=2, label='Momento Óptimo (80%)')
    
    plt.title(f"Ventana de Compra - Cliente {client_id} ({family})")
    fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
    plt.show()
    
    print(f"Intervalo medio real: {intervals.mean():.1f} días")
    print(f"Momento Óptimo Sugerido: {day_optimo:.1f} días desde el último pedido")

# Ejemplo con un cliente leal (Id: 4767)
plot_optimal_window(4767)



## Solución B: Detección de Deterioro Sostenido (CUSUM)
El algoritmo **CUSUM (Cumulative Sum)** detecta cambios pequeños pero persistentes en la media. Es ideal para identificar una "fuga silenciosa" donde el cliente no deja de comprar de golpe, sino que reduce su volumen gradualmente.


In [ ]:

def detect_deterioration_cusum(client_id, family='Anestesia'):
    client_df = df[(df['Id_Cliente'] == client_id) & (df['Familia_Potencial'] == family)]
    daily_sales = client_df.groupby('Fecha')['Valores_H'].sum().resample('ME').sum().fillna(0)
    
    if len(daily_sales) < 6:
        return print("Historial insuficiente para CUSUM")
        
    # Normalizar
    mu = daily_sales.mean()
    std = daily_sales.std()
    if std == 0: std = 1
    
    z = (daily_sales - mu) / std
    
    # CUSUM para caídas (detectar si la media baja)
    k = 0.5 # Sensibilidad (slack)
    h = 3.0 # Umbral de alerta (h * sigma)
    
    s_low = [0]
    for val in z:
        s = min(0, s_low[-1] + val + k)
        s_low.append(s)
    
    s_low = s_low[1:] # quitar el 0 inicial
    
    plt.figure(figsize=(12, 4))
    plt.plot(daily_sales.index, s_low, marker='o', label='CUSUM Score')
    plt.axhline(-h, color='red', linestyle='--', label='Umbral de Alerta')
    plt.fill_between(daily_sales.index, 0, s_low, where=[s < -h for s in s_low], color='red', alpha=0.3)
    plt.title(f"Detección de Deterioro (CUSUM) - Cliente {client_id}")
    plt.ylabel("Desviación Acumulada")
    plt.legend()
    plt.show()

# Ejemplo con un cliente que podría estar en riesgo (Id: 96)
detect_deterioration_cusum(96)



## Solución C: Simulación Dinámica de Stock
Para clientes **Fieles**, el "Momento Óptimo" es justo antes de que el stock llegue a cero. Podemos estimar la "Tasa de Consumo Diario" y simular el nivel de inventario teórico.


In [ ]:

def simulate_stock_level(client_id, family='Anestesia'):
    client_df = df[(df['Id_Cliente'] == client_id) & (df['Familia_Potencial'] == family)]
    
    # Tasa de consumo (Unidades / días entre pedidos)
    dates = client_df['Fecha'].sort_values().unique()
    total_units = client_df['Unidades'].sum()
    total_days = (dates[-1] - dates[0]).days
    
    if total_days == 0: return
    
    burn_rate = total_units / total_days
    
    # Simular desde el último pedido
    last_units = client_df[client_df['Fecha'] == dates[-1]]['Unidades'].sum()
    last_date = dates[-1]
    
    today = df['Fecha'].max()
    days_since = (today - last_date).days
    
    # Proyección a 30 días vista
    projection_days = np.arange(0, days_since + 15)
    stock_level = last_units - (projection_days * burn_rate)
    
    # Limitar stock a no negativo para el plot
    stock_plot = np.clip(stock_level, 0, None)
    
    dates_proj = [last_date + timedelta(days=int(d)) for d in projection_days]
    
    plt.figure(figsize=(10, 5))
    plt.plot(dates_proj, stock_plot, label='Nivel de Stock Estimado', color='green', lw=2)
    plt.axhline(last_units * 0.2, color='orange', linestyle=':', label='Stock de Seguridad (20%)')
    plt.axvline(today, color='blue', linestyle='--', label='Hoy')
    
    # Encontrar cuando cruza el 20%
    safety_threshold = last_units * 0.2
    cross_idx = np.where(stock_level <= safety_threshold)[0]
    if len(cross_idx) > 0:
        opt_date = dates_proj[cross_idx[0]]
        plt.scatter([opt_date], [safety_threshold], color='red', s=100, zorder=5, label='Momento Óptimo')
        print(f"Fecha Óptima de Reposición: {opt_date.date()}")

    plt.title(f"Simulación de Stock - Cliente {client_id}")
    plt.ylabel("Unidades en Stock")
    plt.legend()
    plt.show()

# Ejemplo con cliente leal (Id: 36095)
simulate_stock_level(36095)



## Conclusiones: ¿Cuándo usar cada método?

| Perfil | Método Recomendado | Ventaja |
|---|---|---|
| **Leales** | Simulación de Stock | Máxima precisión para evitar roturas de stock. |
| **Promiscuos** | Probabilidad CDF (Gamma) | Captura el momento estadístico donde suelen comprar. |
| **En Riesgo** | CUSUM | Detecta el cambio de tendencia antes de que el cliente se pierda. |

### Siguientes Pasos
Integrar estos modelos en el `commodities_engine.py` para asignar un **Score de Prioridad Dinámico** basado en la probabilidad de compra y el riesgo de fuga detectado.
